# Exploration des sources de données

Reproduit le chapitre V.II du rapport (figures 11 à 18) : aperçu des fichiers, colonnes, types, et contrôle d'intégrité entre Wikipedia et Kaggle.

Définir `PERFILM_DATA_DIR`, ou laisser `data/raw`. Pour un essai rapide : `python -m perfilm.sample_data data/sample`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from perfilm.config import get_settings
from perfilm.extract import extract
from perfilm.transform.kaggle import transform_kaggle
from perfilm.transform.wiki import transform_wiki

sources = get_settings().sources
raw = extract(sources)
wiki_movies_df = pd.DataFrame(raw.wiki_movies_raw)

## Figures 11 à 13 : premières, médianes et dernières lignes de `wikipedia-movies.json`

In [ ]:
wiki_movies_df.head()

In [ ]:
mid = len(wiki_movies_df) // 2
wiki_movies_df.iloc[mid : mid + 5]

In [ ]:
wiki_movies_df.tail()

## Figure 14 : colonnes du DataFrame et taux de valeurs nulles

In [ ]:
sorted(wiki_movies_df.columns.tolist())

In [ ]:
(wiki_movies_df.isnull().mean() * 100).sort_values().round(1)

## Figure 17 : test de type des variables (listes dans `Box office`)

In [ ]:
box_office = wiki_movies_df["Box office"].dropna()
box_office[box_office.map(lambda x: not isinstance(x, str))]

## Figure 18 : types des variables Kaggle

In [ ]:
raw.kaggle_metadata.dtypes

## Figures 15 et 16 : intégrité entre les deux sources (budget, date de sortie)

In [ ]:
wiki_df = transform_wiki(raw.wiki_movies_raw)
kaggle_df = transform_kaggle(raw.kaggle_metadata)
movies_df = wiki_df.merge(kaggle_df, on="imdb_id", suffixes=("_wiki", "_kaggle"))

movies_df.fillna(0).plot(x="budget_wiki", y="budget_kaggle", kind="scatter")
plt.show()

In [ ]:
movies_df[["release_date_wiki", "release_date_kaggle"]].plot(
    x="release_date_wiki", y="release_date_kaggle", style="."
)
plt.show()